# Pydantic


## Before pydantic


In [14]:
def add_patient_data(name, age):
    if type(name)==str and type(age) == int:
        if(age>0):
            print(name)
            print(age)
            print("Data added successfully to the database!")
        else:
            raise ValueError("Invalid age")
    else:
        raise TypeError("Incorrect data type")
    
def update_patient_data(name: str, age:int):
    if type(name)==str and type(age) == int:
        print(name)
        print(age)
        print("Data update successfully to the database!")
    else:
        raise TypeError("Incorrect data type")

In [15]:
add_patient_data("Jone",25)

Jone
25
Data added successfully to the database!


## After pydantic


In [16]:
from pydantic import BaseModel

class PatientData(BaseModel):
    name:str
    age:int

def add_patient_data(patient:PatientData):
    print(patient.name)
    print(patient.age)
    print("Data added successfully")

def update_patient_data(patient:PatientData):
    print(patient.name)
    print(patient.age)
    print("Data update successfully")

In [21]:
patient_data= {"name":"John","age":25}
patient_1 = PatientData(**patient_data)
add_patient_data(patient_1)

John
25
Data added successfully


In [23]:
patient_data={"name":"Alex","age":28}
patient_2=PatientData(**patient_data)
update_patient_data(patient_2)

Alex
28
Data update successfully


### complex pydantic


In [ ]:
from pydantic import BaseModel
from typing import List, Dict
class PatientData(BaseModel):
    name:str
    age:int
    weight:float
    married:bool
    allergies:List
    contact_info:Dict[str,str]

def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print(patient.contact_info)
    print("Data add successfully")

patient_data={"name":"Alex","age":28,"weight":70.5,"married":True,"allergies":["pollen","dust"],"contact_info":{"email":"alex@example.com"}}
patient_3=PatientData(**patient_data)
add_patient_data(patient_3) 

Alex
28
70.5
True
Data add successfully


optional, default, email, url, custom filters, description, Annotated, strict

In [38]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):
    name:str = Field(...,description="name should be between 2 and 50 characters",max_length=50, min_length=2)
    age:int=Field(...,description="age should be between 0 and 150",gt=0, lt=150)
    email:EmailStr = Field(...,description="email should be a valid email address")
    linkedIn:Optional[AnyUrl]=None
    weight:float = Field(..., strict=True,description="weight should be a positive number",gt=0)
    married:Annotated[bool, Field(description="married should be a boolean value")]=False
    allergies:Annotated[Optional[List[str]], Field(default=None,description="allergies should be a list of strings",max_length=10)] 
    contact_info:Dict[str,str]

def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.linkedIn)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print(patient.contact_info)
    print("Data add successfully")

patient_data={"name":"Alex","age":28,"email":"alex@example.com",
              "linkedIn":"https://www.linkedin.com/in/alex",
              "weight":70.5,
              "contact_info":{"email":"alex@example.com"}}
patient_3=PatientData(**patient_data)
add_patient_data(patient_3) 

Alex
28
alex@example.com
https://www.linkedin.com/in/alex
70.5
False
None
{'email': 'alex@example.com'}
Data add successfully


## Field Validator


In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator
from typing import List, Dict, Optional, Annotated

class PatientData(BaseModel):
    name:str 
    age:int
    email:EmailStr 
    weight:float
    contact_info:Dict[str,str]

    @field_validator('email')
    @classmethod
    def email_validator(cls,email):
        valid_domains=['hdfc.com','icici.com']
        domain_name=email.split('@')[-1]
        if domain_name not in valid_domains:
            raise ValueError(f"Invalid email domain. Allowed domains are: {valid_domains}")

        return email

    @field_validator('name')
    @classmethod
    def name_validator(cls,name):
        return name.upper()
    
def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.weight)
    print(patient.contact_info)
    print("Data add successfully")

patient_data={"name":"Alex","age":28,"email":"alex@icici.com",
              "weight":70.5,
              "contact_info":{"email":"alex@icici.com"}}
patient_3=PatientData(**patient_data)
add_patient_data(patient_3) 

Alex
28
alex@icici.com
70.5
{'email': 'alex@icici.com'}
Data add successfully


## Model Validator

In [56]:
from pydantic import BaseModel, EmailStr, model_validator
from typing import Dict

class PatientData(BaseModel):
    name:str 
    age:int
    email:EmailStr 
    weight:float
    contact_info:Dict[str,str]

    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency' not in model.contact_info:
            raise ValueError("Emergency contact is required for patients above 60 years old.")
        return model

def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print(patient.weight)
    print(patient.contact_info)
    print("Data add successfully")

patient_data={"name":"Alex","age":88,"email":"alex@icici.com",
              "weight":70.5,
              "contact_info":{"email":"alex@icici.com", "emergency":"1234567890"}}
patient_3=PatientData(**patient_data)
add_patient_data(patient_3) 

Alex
88
alex@icici.com
70.5
{'email': 'alex@icici.com', 'emergency': '1234567890'}
Data add successfully


## Computed Fields

In [58]:
from pydantic import BaseModel, EmailStr, computed_field
from typing import Dict

class PatientData(BaseModel):
    name:str 
    age:int
    email:EmailStr 
    weight:float
    heiht:float
    contact_info:Dict[str,str]

    @computed_field
    @property
    def bmi(self) -> float:
       bmi=round(self.weight/(self.heiht**2),2)
       return bmi
    
def add_patient_data(patient: PatientData):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    print("weight",patient.weight)
    print("height",patient.heiht)
    print("bmi",patient.bmi)
    print(patient.contact_info)
    print("Data add successfully")

patient_data={"name":"Alex","age":88,"email":"alex@icici.com",
              "weight":70.5,"heiht":1.65,
              "contact_info":{"email":"alex@icici.com", "emergency":"1234567890"}}
patient_3=PatientData(**patient_data)
add_patient_data(patient_3) 

Alex
88
alex@icici.com
weight 70.5
height 1.65
bmi 25.9
{'email': 'alex@icici.com', 'emergency': '1234567890'}
Data add successfully


## Nested Models

In [59]:
from pydantic import BaseModel
class Address(BaseModel):
    city:str
    state:str
    zip_code:str

class PatientData(BaseModel):
    name:str 
    gender:str
    age:int
    address:Address

address_data={"city":"New York","state":"NY","zip_code":"10001"}
address=Address(**address_data)
patient_data={"name":"Alex","gender":"Male","age":28,"address":address}
patient_3=PatientData(**patient_data)

print(patient_3)
print(patient_3.name)
print(patient_3.address)

name='Alex' gender='Male' age=28 address=Address(city='New York', state='NY', zip_code='10001')
Alex
city='New York' state='NY' zip_code='10001'


## Serialization

In [ ]:
from pydantic import BaseModel
class Address(BaseModel):
    city:str
    state:str
    zip_code:str

class PatientData(BaseModel):
    name:str 
    gender:str
    age:int
    address:Address

address_data={"city":"New York","state":"NY","zip_code":"10001"}
address=Address(**address_data)
patient_data={"name":"Alex","gender":"Male","age":28,"address":address}
patient_3=PatientData(**patient_data)

temp= patient_3.model_dump() # dictionary representation of the model
# temp= patient_3.model_dump_json()
print(temp)
print(type(temp))

{'name': 'Alex', 'gender': 'Male', 'age': 28, 'address': {'city': 'New York', 'state': 'NY', 'zip_code': '10001'}}
<class 'dict'>
